# Inverse Planning in a Simple Gridworld

This notebook is the project demo artifact. It uses the simple task derived from `inv_plan_from_scratch` as the exact inverse-planning baseline, then positions RNN-based models as learned approximations that can later scale to richer settings.

## Table of Contents

### 1. Introduction
States the project goal: infer an agent's latent goal from observed actions in a simple gridworld. Explains why we start from a fully observed exact model before moving to learned sequence models.

### 2. Environment and Task Definition
Defines the gridworld, start state, action space, and candidate goals. This section also clarifies the approximately rational agent assumption used by the exact model.

### 3. Exact Planning and Inverse Planning Baseline
Shows how shortest-path planning induces action values and a Boltzmann policy. Then computes exact posteriors over goals from observed actions to serve as the reference inference method.

### 4. Dataset Construction for Learning
Uses the exact simulator to collect trajectories and supervision targets. Describes the observation encoding and the labels provided to learned models.

### 5. RNN Baselines
Introduces two direct inverse models and one goal-conditioned policy model. The goal is to compare direct prediction against an inverse-imitation-style approach.

### 6. Training Setup
Summarizes the losses, model inputs, and optimization settings used for the RNN experiments. Keeps the notebook concise while leaving the implementation reusable in the package.

### 7. Results and Qualitative Comparison
Compares exact posterior inference against learned approximations on sample trajectories and dataset-level summaries. Highlights where the learned models capture or miss the posterior dynamics.

### 8. Discussion and Future Work
Explains the strengths and limitations of this simplified setup. Connects the notebook to future work with `memo` and richer partially observed models.

In [ ]:
import numpy as np
from IPython.display import HTML, SVG, display

from inverse_planning.data import collect_dataset
from inverse_planning.inference import exact_goal_posterior, online_goal_posteriors
from inverse_planning.simulate import sample_trajectory
from inverse_planning.task import make_default_task
from inverse_planning.visualize import render_gridworld_svg, render_trajectory_frames_html

task = make_default_task(beta=2.0)
task

## 1. Introduction

Inverse planning treats goal inference as an inverse problem: from an observed action sequence, infer which latent goal best explains the behavior. In this notebook, the exact model is small enough to solve directly, which makes it a useful source of ground-truth labels for later RNN experiments.

## 2. Environment and Task Definition

We use a small gridworld with walls, one fixed start location, and a small discrete set of possible goals. The agent is assumed to be approximately rational, meaning it tends to choose actions with higher value under a Boltzmann policy rather than behaving perfectly optimally at every step.

In [ ]:
print('Grid shape:', task.grid.shape)
print('Initial location:', task.init_loc)
print('Goal locations:', task.goal_locs)
print('Action set:', task.actions)
print(task.grid.astype(int))

In [ ]:
display(SVG(render_gridworld_svg(task, title='Simple Gridworld Task')))

### 2.1 Gridworld Layout

The grid encodes free cells and walls, while the start and goal positions define the inference problem. Because the environment is small and deterministic, shortest-path planning is enough to recover action values for the exact baseline.

### 2.2 Agent Assumptions

The agent is modeled as approximately rational rather than perfectly optimal. The rationality coefficient `beta` controls how sharply action probabilities concentrate on high-value actions.

## 3. Exact Planning and Inverse Planning Baseline

The exact baseline computes action probabilities from shortest-path values under each candidate goal. Inference then applies Bayes' rule to recover a posterior distribution over goals from the observed action sequence.

In [ ]:
rng = np.random.default_rng(0)
trajectory = sample_trajectory(task, horizon=8, rng=rng)
print('Sampled goal index:', trajectory.goal_index)
print('Actions:', trajectory.action_indices)
print('Positions:', trajectory.positions)

In [ ]:
display(SVG(render_gridworld_svg(task, trajectory=trajectory, title='Sampled Trajectory')))

### 3.1 Computing Action Values

For each candidate goal, the planner computes how good each available action is at the current state. In this simple task, the value computation reduces to shortest-path distance, making the exact baseline transparent and easy to inspect.

### 3.2 Exact Goal Posterior

Given an observed trajectory, the exact method scores how likely the action sequence is under each candidate goal. Normalizing those scores yields the posterior over goals.

In [ ]:
posterior = exact_goal_posterior(task, trajectory.action_indices)
online = online_goal_posteriors(task, trajectory.action_indices)
print('Final posterior:', np.round(posterior, 3))
print('Online posterior trajectory:')
print(np.round(online, 3))

In [ ]:
display(HTML(render_trajectory_frames_html(task, trajectory, columns=3)))

## 4. Dataset Construction for Learning

The exact simulator is also used to generate training data for RNNs. Each episode provides an action sequence, spatial observation frames, the true latent goal, and the exact online posterior that can be used as a supervision target.

In [ ]:
dataset = collect_dataset(task, n_episodes=16, horizon=8, seed=1)
print('grids:', dataset.grids.shape)
print('actions:', dataset.actions.shape)
print('goals:', dataset.goals.shape)
print('online_posteriors:', dataset.online_posteriors.shape)

### 4.1 Observation Encoding

Each timestep is encoded as a stack of spatial channels containing the wall map, current agent position, goal markers, and a normalized time feature. This format is deliberately chosen so convolutional recurrent models can exploit spatial structure.

### 4.2 Labels from the Exact Model

The dataset stores both the true goal and the exact posterior over goals after each action. That makes it possible to train models either to predict the latent goal directly or to imitate the posterior dynamics of the exact observer.

## 5. RNN Baselines

The project includes three sequence-modeling directions. The first two predict latent state directly, while the third predicts action likelihoods conditioned on a candidate goal and then supports inference by scoring the observed trajectory under each goal.

### 5.1 Action-History GRU

A simple recurrent baseline that consumes encoded observations and previous actions. It provides a low-complexity comparison point for the stronger spatial model.

### 5.2 Convolutional GRU Classifier

A convolutional encoder extracts spatial features before the recurrent update. This is the more principled direct-prediction baseline because it better respects gridworld structure.

### 5.3 Goal-Conditioned Policy RNN

This model is trained to predict actions conditioned on a candidate goal. At test time, each candidate goal can be scored by action likelihood, making the method closer in spirit to inverse planning than direct classification.

## 6. Training Setup

Classifier models are trained with a goal-prediction loss and can optionally match the exact online posterior. The goal-conditioned policy model is trained with action prediction loss, and inference is performed afterward by comparing candidate-goal likelihoods.

In [ ]:
print('Training scripts available:')
print('  python3 scripts/train_goal_belief_rnn.py --dataset data/simple_gridworld.npz --variant conv_gru')
print('  python3 scripts/train_goal_belief_rnn.py --dataset data/simple_gridworld.npz --variant action_gru')
print('  python3 scripts/train_goal_conditioned_policy.py --dataset data/simple_gridworld.npz')

### 6.1 Objectives and Evaluation

The main evaluation targets are final-goal accuracy and how closely model predictions track the exact posterior over time. For the policy model, action likelihood under each candidate goal is the quantity that drives inference.

### 6.2 Practical Note

This notebook keeps training light and delegates full experiments to scripts so the final artifact remains readable. In a final submission, you can either include compact training runs here or load saved checkpoints for the comparison sections.

## 7. Results and Qualitative Comparison

This section is where final experiment outputs should be inserted after training. The intended comparison is exact posterior versus each learned approximation on both individual trajectories and aggregate metrics.

In [ ]:
# Placeholder for final experiment outputs.
# Recommended additions:
# 1. load saved model checkpoints
# 2. run them on held-out trajectories
# 3. print final-goal accuracy and posterior KL divergence
# 4. visualize exact vs predicted posterior curves

### 7.1 What to Show in the Final Version

Include at least one trajectory-level case study where the posterior shifts over time, plus a compact summary table over the test split. That is enough to demonstrate both correctness of the exact baseline and the behavior of the learned approximations.

## 8. Discussion and Future Work

The simple gridworld is a good project starting point because exact inference is still tractable, which gives a clean baseline and a reliable data generator. The main limitation is that this setup does not yet solve the harder belief-space or POMDP inverse-planning problem your instructor warned about.

A natural next step is to keep the same notebook structure while replacing the handcrafted exact planner with a `memo`-based backend for richer planning models. That would preserve the overall experimental story while making the modeling assumptions more realistic.